# Pipeline B: relevance x normalized OvR (fixed formula)

Combines the Stage-1 relevance gatekeeper with the 4 independent one-vs-rest role detectors:

```
P(None | s) = P(z=0 | s)
q_k         = p_pos_k / sum_j(p_pos_j)      (normalized across the 4 legal roles)
P(role_k)   = P(z=1 | s) * q_k
```

Stage 1 is the identical model already analyzed in `03_pipeline_a.ipynb` (section 8) — not repeated here. This notebook focuses on what's different: the 4 OvR detectors and Pipeline B's own combination formula.

## 1. Setup

In [25]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

TEST_DF_PATH = Path("predictions/test_df.csv")
REGISTRY_PATH = Path("roberta_models/best_models_registry.json")
MODELS_DIR = Path("roberta_models")

TEXT_COL = "sent_text"
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ALL_LABELS = ["None", "beoordeling", "beslissing", "materiele feiten", "proceshandelingen"]
LEGAL_LABELS = ["beoordeling", "beslissing", "materiele feiten", "proceshandelingen"]
OVR_MODEL_KEYS = ["ovr_beoordeling", "ovr_beslissing", "ovr_materiele_feiten", "ovr_proceshandelingen"]

print("Device:", DEVICE)

Device: cuda


## 2. Load test data

In [26]:
test_df = pd.read_csv(TEST_DF_PATH, keep_default_na=False)

print("Test rows:", len(test_df))
test_df["label"].value_counts()

Test rows: 1502


label
None                 541
beoordeling          389
materiele feiten     297
proceshandelingen    206
beslissing            69
Name: count, dtype: int64

## 3. Load Stage 1 and the 4 OvR models

In [27]:
with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
    registry = json.load(f)


def clean_col(label):
    return label.strip().lower().replace(" ", "_")


def resolve_model_path(model_info):
    return MODELS_DIR / Path(model_info["saved_path"]).name


def load_model(model_key):
    model_info = registry[model_key]
    model_path = resolve_model_path(model_info)

    if not model_path.exists():
        raise FileNotFoundError(f"Model path not found: {model_path}")

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(DEVICE)
    model.eval()

    labels = model_info.get("labels_order", model_info.get("labels"))
    max_len = model_info.get("max_len", 256)

    return tokenizer, model, labels, max_len


@torch.no_grad()
def predict_probs(texts, tokenizer, model, max_len, batch_size=BATCH_SIZE, desc=""):
    all_probs = []

    for start in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[start:start + batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)
        all_probs.append(probs.cpu().numpy())

    return np.vstack(all_probs)

## 4. Run inference

In [28]:
texts = test_df[TEXT_COL].fillna("").astype(str).tolist()

tokenizer1, model1, labels1, max_len1 = load_model("stage1_gatekeeper")
stage1_probs = predict_probs(texts, tokenizer1, model1, max_len1, desc="Stage 1 (relevance)")

test_df["p_none"] = stage1_probs[:, 0]
test_df["p_labelled"] = stage1_probs[:, 1]

del model1, tokenizer1
print("Stage 1 labels order:", labels1)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Stage 1 (relevance):   0%|          | 0/47 [00:00<?, ?it/s]

Stage 1 labels order: ['not_labelled', 'labelled']


In [29]:
for model_key in OVR_MODEL_KEYS:
    tokenizer, model, labels, max_len = load_model(model_key)
    probs = predict_probs(texts, tokenizer, model, max_len, desc=model_key)

    pos_label = clean_col(registry[model_key]["pos_label"])
    test_df[f"{pos_label}_p_not"] = probs[:, 0]
    test_df[f"{pos_label}_p_pos"] = probs[:, 1]

    del model, tokenizer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print("OvR detectors run:", OVR_MODEL_KEYS)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ovr_beoordeling:   0%|          | 0/47 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ovr_beslissing:   0%|          | 0/47 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ovr_materiele_feiten:   0%|          | 0/47 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ovr_proceshandelingen:   0%|          | 0/47 [00:00<?, ?it/s]

OvR detectors run: ['ovr_beoordeling', 'ovr_beslissing', 'ovr_materiele_feiten', 'ovr_proceshandelingen']


## 5. Save raw model outputs

In [30]:
OUTPUT_PATH = Path("predictions/pipeline_b_test_outputs.csv")
test_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Saved:", OUTPUT_PATH)

Saved: predictions\pipeline_b_test_outputs.csv


## 6. Combine: Pipeline B formula

In [31]:
def make_pipeline_b(df, eps=1e-12):
    ovr_scores = np.column_stack([
        df[f"{clean_col(l)}_p_pos"].astype(float).values for l in LEGAL_LABELS
    ])
    ovr_scores = np.clip(ovr_scores, eps, 1.0)

    q = ovr_scores / ovr_scores.sum(axis=1, keepdims=True)

    final_probs = np.column_stack([
        df["p_none"].astype(float).values,
        df["p_labelled"].astype(float).values[:, None] * q
    ])

    return pd.DataFrame(final_probs, columns=ALL_LABELS, index=df.index)


pipeline_b_probs = make_pipeline_b(test_df)
test_df["pipeline_b_pred_label"] = pipeline_b_probs.idxmax(axis=1)

pipeline_b_probs.head()

,None,beoordeling,beslissing,materiele feiten,proceshandelingen
0,0.666001,0.319407,0.000069,0.013809,0.000714
1,0.195918,0.797403,0.000373,0.004270,0.002037
2,0.028706,0.003467,0.000210,0.961156,0.006461
3,0.033778,0.004968,0.000466,0.956823,0.003965
4,0.946429,0.013067,0.000073,0.038481,0.001951


## 7. Evaluate

Sanity check: should reproduce the previously reported Pipeline B numbers (accuracy 0.6372, macro F1 0.6604).

In [32]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

y_true = test_df["label"]
y_pred = test_df["pipeline_b_pred_label"]

acc = accuracy_score(y_true, y_pred)
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=ALL_LABELS, average="macro", zero_division=0
)

print(f"Pipeline B accuracy: {acc:.4f}")
print(f"Macro P: {macro_p:.4f}  Macro R: {macro_r:.4f}  Macro F1: {macro_f1:.4f}")
print()
print(classification_report(y_true, y_pred, labels=ALL_LABELS, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=ALL_LABELS)
pd.DataFrame(
    cm,
    index=[f"true_{l}" for l in ALL_LABELS],
    columns=[f"pred_{l}" for l in ALL_LABELS]
)

Pipeline B accuracy: 0.6372
Macro P: 0.6705  Macro R: 0.6567  Macro F1: 0.6604

                   precision    recall  f1-score   support

             None     0.7318    0.6303    0.6773       541
      beoordeling     0.5723    0.7326    0.6426       389
       beslissing     0.9062    0.8406    0.8722        69
 materiele feiten     0.6022    0.5556    0.5779       297
proceshandelingen     0.5400    0.5243    0.5320       206

         accuracy                         0.6372      1502
        macro avg     0.6705    0.6567    0.6604      1502
     weighted avg     0.6466    0.6372    0.6377      1502



,pred_None,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_None,341,114,4,46,36
true_beoordeling,46,285,2,36,20
true_beslissing,7,2,58,2,0
true_materiele feiten,58,38,0,165,36
true_proceshandelingen,14,59,0,25,108


## 8. OvR detector error analysis

Each of the 4 detectors evaluated independently on the 961 gold-relevant sentences (they were trained only on that subset, same as Stage 2).

In [33]:
legal_df = test_df[test_df["label"].isin(LEGAL_LABELS)].copy()

ovr_results = []

for label in LEGAL_LABELS:
    clean = clean_col(label)
    y_t = (legal_df["label"] == label).astype(int)
    y_p = (legal_df[f"{clean}_p_pos"] > legal_df[f"{clean}_p_not"]).astype(int)

    tp = ((y_t == 1) & (y_p == 1)).sum()
    fp = ((y_t == 0) & (y_p == 1)).sum()
    fn = ((y_t == 1) & (y_p == 0)).sum()
    tn = ((y_t == 0) & (y_p == 0)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    ovr_results.append({
        "label": label, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)
    })

pd.DataFrame(ovr_results)

,label,TP,FP,FN,TN,precision,recall,f1
0,beoordeling,313,98,76,474,0.7616,0.8046,0.7825
1,beslissing,62,2,7,890,0.9688,0.8986,0.9323
2,materiele feiten,200,59,97,605,0.7722,0.6734,0.7194
3,proceshandelingen,110,52,96,703,0.6790,0.5340,0.5978


### 8.1 How often do the 4 detectors agree, conflict, or all stay silent?

This is the assumption Pipeline B's normalization step papers over — it always forces the 4 scores to sum to 1, whether zero, one, or several detectors actually fired.

In [34]:
ovr_pos_cols = {l: f"{clean_col(l)}_p_pos" for l in LEGAL_LABELS}
ovr_not_cols = {l: f"{clean_col(l)}_p_not" for l in LEGAL_LABELS}

vote_matrix = pd.DataFrame({
    l: (legal_df[ovr_pos_cols[l]] > legal_df[ovr_not_cols[l]]).astype(int)
    for l in LEGAL_LABELS
}, index=legal_df.index)

legal_df["n_ovr_positive"] = vote_matrix.sum(axis=1)

vote_counts = legal_df["n_ovr_positive"].value_counts().sort_index()
vote_pct = (100 * vote_counts / len(legal_df)).round(2)
print("Distribution of positive-vote counts on the 961 gold-relevant sentences")
display(pd.DataFrame({"count": vote_counts, "pct": vote_pct}))

# Raw-score argmax among the 4 detectors, regardless of whether any cleared 0.5
ovr_raw = legal_df[[ovr_pos_cols[l] for l in LEGAL_LABELS]].copy()
ovr_raw.columns = LEGAL_LABELS
legal_df["ovr_only_pred_label"] = ovr_raw.idxmax(axis=1)
legal_df["ovr_only_correct"] = legal_df["ovr_only_pred_label"] == legal_df["label"]

print("\nOvR argmax accuracy, broken down by how many detectors actually fired:")
display(
    legal_df.groupby("n_ovr_positive")["ovr_only_correct"]
    .agg(n="count", accuracy="mean")
    .assign(accuracy=lambda d: d["accuracy"].round(3))
)

Distribution of positive-vote counts on the 961 gold-relevant sentences


,count,pct
n_ovr_positive,,
0,108,11.24
1,810,84.29
2,43,4.47



OvR argmax accuracy, broken down by how many detectors actually fired:


,n,accuracy
n_ovr_positive,,
0,108,0.352
1,810,0.795
2,43,0.581


## 9. Where do Pipeline B's errors actually come from?

The same structural argument as Pipeline A applies here, for a slightly different reason. Normalizing the 4 OvR scores (`q_k = p_pos_k / sum`) divides every one of them by the *same* sentence-specific constant, and multiplying by `p_labelled` afterward applies another constant across all 4 — neither step can change which of the 4 roles has the highest score. So **whenever Pipeline B predicts a role, it is always exactly the OvR detector with the highest raw positive-class score**, regardless of Stage 1 and regardless of normalization. That gives the same clean three-way split as Pipeline A.

In [35]:
# Confirm the claim empirically before relying on it.

ovr_raw_full = test_df[[ovr_pos_cols[l] for l in LEGAL_LABELS]].copy()
ovr_raw_full.columns = LEGAL_LABELS
test_df["ovr_only_pred_label"] = ovr_raw_full.idxmax(axis=1)

role_predicted_mask = test_df["pipeline_b_pred_label"] != "None"
mismatch = test_df[
    role_predicted_mask & (test_df["pipeline_b_pred_label"] != test_df["ovr_only_pred_label"])
]

print(f"Rows where Pipeline B predicts a role that differs from the OvR argmax: {len(mismatch)}")
assert len(mismatch) == 0, "Unexpected: Pipeline B's role choice diverged from the OvR argmax"

Rows where Pipeline B predicts a role that differs from the OvR argmax: 0


In [36]:
def decompose_pipeline_b_error(row):
    if row["label"] == row["pipeline_b_pred_label"]:
        return "pipeline_correct"
    if row["p_labelled"] <= row["p_none"]:
        return "inherited_from_stage1"
    if row["ovr_only_pred_label"] != row["label"]:
        return "inherited_from_ovr"
    return "combination_induced"


legal_df["pipeline_b_pred_label"] = test_df.loc[legal_df.index, "pipeline_b_pred_label"]
legal_df["pipeline_b_error_type"] = legal_df.apply(decompose_pipeline_b_error, axis=1)

decomposition_b = legal_df["pipeline_b_error_type"].value_counts()
decomposition_b_pct = (100 * decomposition_b / len(legal_df)).round(2)

print(f"Pipeline B error decomposition on the {len(legal_df)} gold-relevant test sentences")
decomp_b_table = pd.DataFrame({"count": decomposition_b, "pct_of_gold_relevant": decomposition_b_pct})
print(decomp_b_table)

Pipeline B error decomposition on the 961 gold-relevant test sentences
                       count  pct_of_gold_relevant
pipeline_b_error_type                             
pipeline_correct         616                 64.10
inherited_from_ovr       227                 23.62
inherited_from_stage1    110                 11.45
combination_induced        8                  0.83


In [37]:
# The interesting category: Stage 1 says relevant, the OvR argmax already matches
# the true role, yet the combination formula still flips the decision to None.

combo_induced_b = legal_df[legal_df["pipeline_b_error_type"] == "combination_induced"].copy()

print(f"Combination-induced errors: {len(combo_induced_b)}")
if len(combo_induced_b) > 0:
    sample = combo_induced_b.sample(min(6, len(combo_induced_b)), random_state=42)
    for _, row in sample.iterrows():
        true_role = row["label"]
        combined_p = row["p_labelled"] * (
            row[ovr_pos_cols[true_role]] / sum(row[ovr_pos_cols[l]] for l in LEGAL_LABELS)
        )
        print(
            f"- [{row['case_name']} | hdr={row['hdr_group']} | true={true_role} | "
            f"p_none={row['p_none']:.2f} vs combined={combined_p:.2f} | n_positive_votes={row['n_ovr_positive']}] "
            f"{row['sent_text']}"
        )

Combination-induced errors: 8
- [ECLI:NL:RBOBR:2020:3584.txt | hdr=Feiten | true=beoordeling | p_none=0.48 vs combined=0.45 | n_positive_votes=1] .
- [ECLI:NL:RBOVE:2017:1505.txt | hdr=Feiten | true=materiele feiten | p_none=0.42 vs combined=0.30 | n_positive_votes=0] Bovendien had u die dag geen vervolgplanning meer.
- [ECLI:NL:RBNNE:2016:4308.txt | hdr=Proceshandelingen partijen | true=proceshandelingen | p_none=0.47 vs combined=0.37 | n_positive_votes=1] Met betrekking tot hen heeft de oom voorts bepaald dat bij vooroverlijden van een broer of (schoon)zus met achterlating van een echtgeno(o)t(e) deze echtgeno(o)t(e) voor de voor overleden broer of (schoon)zus in de plaats treedt.
- [ECLI:NL:RBDHA:2021:6182.txt | hdr=Beoordeling | true=materiele feiten | p_none=0.36 vs combined=0.34 | n_positive_votes=0] Het betreft een speciale regeling die voorgaat op de Wob (een lex specialis).
- [ECLI:NL:RBOBR:2020:3584.txt | hdr=Proceshandelingen partijen | true=proceshandelingen | p_none=0.48 v

## 10. Pipeline A vs. Pipeline B: error decomposition side by side

In [38]:
# Pipeline A numbers from 03_pipeline_a.ipynb, section 10 (already verified).
pipeline_a_decomp = {
    "pipeline_correct": 615,
    "inherited_from_stage1": 110,
    "inherited_from_stage2_or_ovr": 230,
    "combination_induced": 6
}

pipeline_b_decomp = {
    "pipeline_correct": int(decomposition_b.get("pipeline_correct", 0)),
    "inherited_from_stage1": int(decomposition_b.get("inherited_from_stage1", 0)),
    "inherited_from_stage2_or_ovr": int(decomposition_b.get("inherited_from_ovr", 0)),
    "combination_induced": int(decomposition_b.get("combination_induced", 0))
}

decomp_comparison = pd.DataFrame([
    {"category": k, "Pipeline A": pipeline_a_decomp[k], "Pipeline B": pipeline_b_decomp[k]}
    for k in pipeline_a_decomp
])
decomp_comparison["Pipeline A %"] = (100 * decomp_comparison["Pipeline A"] / 961).round(1)
decomp_comparison["Pipeline B %"] = (100 * decomp_comparison["Pipeline B"] / 961).round(1)

print(decomp_comparison)

                       category  Pipeline A  Pipeline B  Pipeline A %  \
0              pipeline_correct         615         616          64.0   
1         inherited_from_stage1         110         110          11.4   
2  inherited_from_stage2_or_ovr         230         227          23.9   
3           combination_induced           6           8           0.6   

   Pipeline B %  
0          64.1  
1          11.4  
2          23.6  
3           0.8  


## 11. Pipeline B vs. Pipeline A vs. direct 5-way: relevance errors vs. role-confusion errors

In [39]:
def classify_error(row, pred_col):
    if row["label"] == row[pred_col]:
        return "correct"
    if row["label"] == "None" or row[pred_col] == "None":
        return "relevance_error"
    return "role_confusion_error"


# Pipeline B, computed here
test_df["pipeline_b_error_type_5way"] = test_df.apply(classify_error, pred_col="pipeline_b_pred_label", axis=1)
b_errors = test_df[test_df["pipeline_b_error_type_5way"] != "correct"]
b_split_pct = (100 * b_errors["pipeline_b_error_type_5way"].value_counts() / len(b_errors)).round(2)

# Pipeline A: reload its raw saved outputs and recompute the combination formula
# (pipeline_a_test_outputs.csv was saved before the formula was applied, so it only
# has the raw Stage 1 / Stage 2 columns, not the final pipeline_a_pred_label).
pipeline_a_df = pd.read_csv("predictions/pipeline_a_test_outputs.csv", keep_default_na=False)

a_probs = pd.DataFrame(index=pipeline_a_df.index)
a_probs["None"] = pipeline_a_df["p_none"]
for label in LEGAL_LABELS:
    a_probs[label] = pipeline_a_df["p_labelled"] * pipeline_a_df[f"stage2_p_{clean_col(label)}"]
pipeline_a_df["pipeline_a_pred_label"] = a_probs.idxmax(axis=1)

pipeline_a_df["error_type_5way"] = pipeline_a_df.apply(classify_error, pred_col="pipeline_a_pred_label", axis=1)
a_errors = pipeline_a_df[pipeline_a_df["error_type_5way"] != "correct"]
a_split_pct = (100 * a_errors["error_type_5way"].value_counts() / len(a_errors)).round(2)

comparison = pd.DataFrame({
    "system": ["Direct 5-way", "Pipeline A", "Pipeline B"],
    "total_errors": [502, len(a_errors), len(b_errors)],
    "relevance_error_pct": [
        60.6,
        a_split_pct.get("relevance_error", 0.0),
        b_split_pct.get("relevance_error", 0.0)
    ],
    "role_confusion_pct": [
        39.4,
        a_split_pct.get("role_confusion_error", 0.0),
        b_split_pct.get("role_confusion_error", 0.0)
    ]
})

print(comparison)

         system  total_errors  relevance_error_pct  role_confusion_pct
0  Direct 5-way           502                60.60               39.40
1    Pipeline A           545                58.53               41.47
2    Pipeline B           545                59.63               40.37


## 11b. Document-level error clustering (Pipeline B's combined prediction)

Same methodology as `01_five_way_inference.ipynb` (section 7, document-level error clustering), computed here on Pipeline B's final combined prediction (`pipeline_b_pred_label`) rather than a single component model's own errors, so the two are directly comparable.

In [40]:
pipeline_b_case_error_counts = (
    test_df
    .assign(is_error=test_df["label"] != test_df["pipeline_b_pred_label"])
    .groupby("case_name")["is_error"]
    .agg(n_sentences="count", n_errors="sum")
)
pipeline_b_case_error_counts["error_rate_pct"] = (
    100 * pipeline_b_case_error_counts["n_errors"] / pipeline_b_case_error_counts["n_sentences"]
).round(2)

print(f"Test-set documents: {len(pipeline_b_case_error_counts)}")
print(f"Documents with zero errors: {(pipeline_b_case_error_counts['n_errors'] == 0).sum()}")

sorted_by_errors = pipeline_b_case_error_counts.sort_values("n_errors", ascending=False)
total_errors = sorted_by_errors["n_errors"].sum()
cum_pct = 100 * sorted_by_errors["n_errors"].cumsum() / total_errors

print("\nCumulative share of all errors covered by the top-N error-heaviest documents:")
for n in [5, 10, 20]:
    print(f"  top {n} documents: {cum_pct.iloc[n - 1]:.1f}%")

print("\nTop 10 documents by error count:")
sorted_by_errors.head(10)

Test-set documents: 20
Documents with zero errors: 2

Cumulative share of all errors covered by the top-N error-heaviest documents:
  top 5 documents: 50.8%
  top 10 documents: 80.6%
  top 20 documents: 100.0%

Top 10 documents by error count:


,n_sentences,n_errors,error_rate_pct
case_name,,,
ECLI:NL:RBOBR:2020:3584.txt,221,74,33.48
ECLI:NL:RBOVE:2017:1505.txt,129,59,45.74
ECLI:NL:HR:2018:874.txt,108,50,46.30
ECLI:NL:RBDHA:2018:3316.txt,121,49,40.50
ECLI:NL:RBNNE:2015:2927.txt,99,45,45.45
ECLI:NL:RBNNE:2016:4308.txt,77,36,46.75
ECLI:NL:RBOBR:2016:6963.txt,123,35,28.46
ECLI:NL:GHARL:2017:8427.txt,148,33,22.30
ECLI:NL:GHARL:2015:6258.txt,57,29,50.88


## 12. Header-prior Bayesian fusion

Header fusion applied independently to Stage 1 and each of the 4 OvR detectors — each gets its own validation-tuned λ (see `06_stage1_relevance.ipynb` and `08_ovr_detectors.ipynb` for the component-level results this reuses) — then recombined through the same Pipeline B normalize-and-multiply formula used in section 6.

In [41]:
from sklearn.metrics import precision_recall_fscore_support

TRAIN_DF_PATH = Path("predictions/train_df.csv")
EVAL_DF_PATH = Path("predictions/eval_df.csv")
ALPHA = 1.0
BINARY_CLASSES = [0, 1]
LAMBDA_GRID = [0, 0.1, 0.25, 0.5, 1, 2, 3, 5]

train_df = pd.read_csv(TRAIN_DF_PATH, keep_default_na=False)
eval_df = pd.read_csv(EVAL_DF_PATH, keep_default_na=False)
train_df["y_labelled"] = (train_df["label"] != "None").astype(int)
eval_df["y_labelled"] = (eval_df["label"] != "None").astype(int)
train_legal = train_df[train_df["label"].isin(LEGAL_LABELS)].copy()
eval_legal = eval_df[eval_df["label"].isin(LEGAL_LABELS)].copy()
legal_df = test_df[test_df["label"].isin(LEGAL_LABELS)].copy()


def make_header_prior(df_train, label_col, labels, alpha=1.0):
    counts = (
        df_train.groupby(["hdr_group", label_col]).size()
        .unstack(fill_value=0)
        .reindex(columns=labels, fill_value=0)
    )
    probs = counts + alpha
    return probs.div(probs.sum(axis=1), axis=0)


def make_global_prior(df_train, label_col, labels, alpha=1.0):
    counts = df_train[label_col].value_counts().reindex(labels, fill_value=0) + alpha
    return (counts / counts.sum()).values


def get_meta_prior_df(df_apply, header_prior_train, global_prior, labels):
    def get_prior(hdr_group):
        if hdr_group in header_prior_train.index:
            return header_prior_train.loc[hdr_group].values
        return global_prior

    mat = np.vstack(df_apply["hdr_group"].apply(get_prior))
    return pd.DataFrame(mat, columns=labels, index=df_apply.index)


def combine_log_scores(text_probs, metadata_probs, lam, eps=1e-12):
    text = np.clip(text_probs.values, eps, 1.0)
    metadata = np.clip(metadata_probs.values, eps, 1.0)

    scores = np.log(text) + lam * np.log(metadata)
    scores -= scores.max(axis=1, keepdims=True)
    scores = np.exp(scores)
    scores /= scores.sum(axis=1, keepdims=True)

    return pd.DataFrame(scores, columns=text_probs.columns, index=text_probs.index)


print("Train rows:", len(train_df), " | Validation rows:", len(eval_df))

Train rows: 5608  | Validation rows: 1281


### 12.1 Tune λ for Stage 1

In [42]:
tok1, mdl1, lbl1, ml1 = load_model("stage1_gatekeeper")
eval_stage1_probs = predict_probs(
    eval_df[TEXT_COL].fillna("").astype(str).tolist(), tok1, mdl1, ml1, BATCH_SIZE, desc="[val] stage1"
)
del mdl1, tok1
if DEVICE == "cuda":
    torch.cuda.empty_cache()

eval_stage1_text_probs = pd.DataFrame(eval_stage1_probs, columns=BINARY_CLASSES, index=eval_df.index)

header_prior_s1 = make_header_prior(train_df, "y_labelled", BINARY_CLASSES, ALPHA)
global_prior_s1 = make_global_prior(train_df, "y_labelled", BINARY_CLASSES, ALPHA)
eval_meta_prior_s1 = get_meta_prior_df(eval_df, header_prior_s1, global_prior_s1, BINARY_CLASSES)

sweep_s1 = []
for lam in LAMBDA_GRID:
    fused = combine_log_scores(eval_stage1_text_probs, eval_meta_prior_s1, lam)
    pred = fused.idxmax(axis=1)
    _, _, f1, _ = precision_recall_fscore_support(
        eval_df["y_labelled"], pred, labels=BINARY_CLASSES, average="macro", zero_division=0
    )
    sweep_s1.append({"lambda": lam, "val_macro_f1": round(f1, 4)})

sweep_s1_df = pd.DataFrame(sweep_s1)
best_lambda_s1 = sweep_s1_df.loc[sweep_s1_df["val_macro_f1"].idxmax(), "lambda"]
print(sweep_s1_df)
print(f"Best Stage 1 lambda: {best_lambda_s1}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[val] stage1:   0%|          | 0/41 [00:00<?, ?it/s]

   lambda  val_macro_f1
0    0.00        0.7648
1    0.10        0.7614
2    0.25        0.7557
3    0.50        0.7439
4    1.00        0.7100
5    2.00        0.6578
6    3.00        0.6229
7    5.00        0.5191
Best Stage 1 lambda: 0.0


### 12.2 Tune λ for each of the 4 OvR detectors

In [43]:
ovr_header_priors = {}
ovr_global_priors = {}
ovr_best_lambda = {}

for label in LEGAL_LABELS:
    clean = clean_col(label)
    model_key = f"ovr_{clean}"

    train_legal[f"target_{clean}"] = (train_legal["label"] == label).astype(int)
    eval_legal_target = (eval_legal["label"] == label).astype(int)

    tokenizer, model, _, max_len = load_model(model_key)
    eval_probs = predict_probs(
        eval_legal[TEXT_COL].fillna("").astype(str).tolist(),
        tokenizer, model, max_len, BATCH_SIZE, desc=f"[val] {model_key}"
    )
    del model, tokenizer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    eval_text_probs = pd.DataFrame(eval_probs, columns=BINARY_CLASSES, index=eval_legal.index)

    header_prior = make_header_prior(train_legal, f"target_{clean}", BINARY_CLASSES, ALPHA)
    global_prior = make_global_prior(train_legal, f"target_{clean}", BINARY_CLASSES, ALPHA)
    eval_meta_prior = get_meta_prior_df(eval_legal, header_prior, global_prior, BINARY_CLASSES)

    sweep = []
    for lam in LAMBDA_GRID:
        fused = combine_log_scores(eval_text_probs, eval_meta_prior, lam)
        pred = fused.idxmax(axis=1)
        _, _, f1, _ = precision_recall_fscore_support(
            eval_legal_target, pred, labels=BINARY_CLASSES, average="binary", pos_label=1, zero_division=0
        )
        sweep.append({"lambda": lam, "val_f1": round(f1, 4)})
    sweep_df = pd.DataFrame(sweep)
    best_lambda = sweep_df.loc[sweep_df["val_f1"].idxmax(), "lambda"]

    ovr_header_priors[label] = header_prior
    ovr_global_priors[label] = global_prior
    ovr_best_lambda[label] = best_lambda

    print(f"{label}: best lambda = {best_lambda}")
    print(sweep_df.to_string(index=False))
    print()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[val] ovr_beoordeling:   0%|          | 0/24 [00:00<?, ?it/s]

beoordeling: best lambda = 2.0
 lambda  val_f1
   0.00  0.8365
   0.10  0.8379
   0.25  0.8379
   0.50  0.8401
   1.00  0.8447
   2.00  0.8498
   3.00  0.8484
   5.00  0.8447



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[val] ovr_beslissing:   0%|          | 0/24 [00:00<?, ?it/s]

...

[val] ovr_materiele_feiten:   0%|          | 0/24 [00:00<?, ?it/s]

materiele feiten: best lambda = 2.0
 lambda  val_f1
   0.00  0.7034
   0.10  0.6932
   0.25  0.7012
   0.50  0.7160
   1.00  0.7398
   2.00  0.7452
   3.00  0.6747
   5.00  0.5871



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[val] ovr_proceshandelingen:   0%|          | 0/24 [00:00<?, ?it/s]

proceshandelingen: best lambda = 0.1
 lambda  val_f1
   0.00  0.6467
   0.10  0.6566
   0.25  0.6565
   0.50  0.6562
   1.00  0.6558
   2.00  0.6408
   3.00  0.5736
   5.00  0.2673



### 12.3 Apply the fused stages to the test set and recombine via Pipeline B

In [44]:
test_stage1_text_probs = test_df[["p_none", "p_labelled"]].copy()
test_stage1_text_probs.columns = BINARY_CLASSES
test_meta_prior_s1 = get_meta_prior_df(test_df, header_prior_s1, global_prior_s1, BINARY_CLASSES)


def fuse_ovr_for_test(lam_per_label):
    fused_pos = pd.DataFrame(index=test_df.index)
    for label in LEGAL_LABELS:
        clean = clean_col(label)
        text_probs = test_df[[f"{clean}_p_not", f"{clean}_p_pos"]].copy()
        text_probs.columns = BINARY_CLASSES
        meta_prior = get_meta_prior_df(
            test_df, ovr_header_priors[label], ovr_global_priors[label], BINARY_CLASSES
        )
        lam = lam_per_label[label]
        fused = combine_log_scores(text_probs, meta_prior, lam)
        fused_pos[label] = fused[1]
    return fused_pos


pipeline_b_header_results = []

for lam_s1, lam_ovr, name in [
    (0, {l: 0 for l in LEGAL_LABELS}, "no fusion"),
    (1, {l: 1 for l in LEGAL_LABELS}, "both lambda=1"),
    (best_lambda_s1, ovr_best_lambda, "tuned (per-component)")
]:
    fused_s1 = combine_log_scores(test_stage1_text_probs, test_meta_prior_s1, lam_s1)
    fused_ovr_pos = fuse_ovr_for_test(lam_ovr)

    eps = 1e-12
    ovr_scores = np.clip(fused_ovr_pos.values, eps, 1.0)
    q = ovr_scores / ovr_scores.sum(axis=1, keepdims=True)

    final_probs = np.column_stack([fused_s1[0].values, fused_s1[1].values[:, None] * q])
    final_probs_df = pd.DataFrame(final_probs, columns=ALL_LABELS, index=test_df.index)
    pred = final_probs_df.idxmax(axis=1)

    acc = accuracy_score(test_df["label"], pred)
    _, _, f1, _ = precision_recall_fscore_support(
        test_df["label"], pred, labels=ALL_LABELS, average="macro", zero_division=0
    )
    pipeline_b_header_results.append({
        "setting": name, "lambda_stage1": lam_s1,
        "test_accuracy": round(acc, 4), "test_macro_f1": round(f1, 4)
    })

print(pd.DataFrame(pipeline_b_header_results))

                 setting  lambda_stage1  test_accuracy  test_macro_f1
0              no fusion            0.0         0.6372         0.6604
1          both lambda=1            1.0         0.6411         0.6717
2  tuned (per-component)            0.0         0.6551         0.6810


In [45]:
print("Full classification report at tuned per-component lambda (test set)")
print(classification_report(test_df["label"], pred, labels=ALL_LABELS, digits=4, zero_division=0))

Full classification report at tuned per-component lambda (test set)
                   precision    recall  f1-score   support

             None     0.7440    0.6285    0.6814       541
      beoordeling     0.5962    0.7326    0.6574       389
       beslissing     0.9062    0.8406    0.8722        69
 materiele feiten     0.6538    0.5724    0.6104       297
proceshandelingen     0.5391    0.6359    0.5835       206

         accuracy                         0.6551      1502
        macro avg     0.6879    0.6820    0.6810      1502
     weighted avg     0.6672    0.6551    0.6565      1502



## 13. Confidence thresholding (None-first, then per-detector thresholds)

Unlike the 5-way model and Pipeline A, Pipeline B's role decision comes from 4 independent OvR detectors rather than one mutually-exclusive softmax — so per your direction, each detector gets its own tuned threshold rather than relying on a plain argmax. Decision rule, in order:

1. If Stage 1's `P(None) ≥ θ_none` → predict None.
2. Otherwise, check each OvR detector's positive-class probability against its own `θ_class`. If none clear their threshold → predict None (fallback). If one or more clear, predict whichever has the highest positive-class probability among those that cleared.

Every threshold here is tuned on the training set against **its own class's binary F1, in isolation** — not against the downstream pipeline's overall macro F1. `θ_none` is chosen purely from Stage 1's own None-vs-rest separation (independent of the OvR stage entirely), and each `θ_class` is chosen purely from that one detector's own positive-class separation. The precision-maximizing alternative is shown alongside each for comparison.

### 13.1 Tune each OvR detector's own threshold (train, legal-labelled subset)

In [46]:
THETA_GRID = np.round(np.arange(0.05, 1.0, 0.05), 2)

train_legal_ovr_probs = {}
ovr_theta_class = {}

for label in LEGAL_LABELS:
    clean = clean_col(label)
    model_key = f"ovr_{clean}"

    tokenizer, model, _, max_len = load_model(model_key)
    probs = predict_probs(
        train_legal[TEXT_COL].fillna("").astype(str).tolist(),
        tokenizer, model, max_len, BATCH_SIZE, desc=f"[train-legal] {model_key}"
    )
    del model, tokenizer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    p_pos = pd.Series(probs[:, 1], index=train_legal.index)
    train_legal_ovr_probs[label] = p_pos
    target = (train_legal["label"] == label).astype(int)

    sweep = []
    for theta in THETA_GRID:
        pred = (p_pos >= theta).astype(int)
        precision, recall, f1, _ = precision_recall_fscore_support(
            target, pred, labels=[0, 1], average="binary", pos_label=1, zero_division=0
        )
        sweep.append({
            "theta": theta,
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1": round(f1, 4)
        })
    sweep_df = pd.DataFrame(sweep)
    best_theta_f1 = sweep_df.loc[sweep_df["f1"].idxmax(), "theta"]
    best_theta_precision = sweep_df.loc[sweep_df["precision"].idxmax(), "theta"]
    ovr_theta_class[label] = best_theta_f1

    print(f"{label}: best theta (F1) = {best_theta_f1} (F1={sweep_df['f1'].max():.4f})  |  best theta (precision) = {best_theta_precision} (P={sweep_df['precision'].max():.4f})")

print("\nPer-detector thresholds in use (F1-maximizing):", ovr_theta_class)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[train-legal] ovr_beoordeling:   0%|          | 0/113 [00:00<?, ?it/s]

beoordeling: best theta (F1) = 0.45 (F1=0.9429)  |  best theta (precision) = 0.95 (P=0.9774)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[train-legal] ovr_beslissing:   0%|          | 0/113 [00:00<?, ?it/s]

...

[train-legal] ovr_materiele_feiten:   0%|          | 0/113 [00:00<?, ?it/s]

materiele feiten: best theta (F1) = 0.55 (F1=0.8943)  |  best theta (precision) = 0.95 (P=0.9865)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[train-legal] ovr_proceshandelingen:   0%|          | 0/113 [00:00<?, ?it/s]

proceshandelingen: best theta (F1) = 0.3 (F1=0.8926)  |  best theta (precision) = 0.95 (P=0.9881)

Per-detector thresholds in use (F1-maximizing): {'beoordeling': 0.45, 'beslissing': 0.15, 'materiele feiten': 0.55, 'proceshandelingen': 0.3}


### 13.2 Tune θ_none on the full training set (thresholds from 13.1 held fixed)

In [47]:
tok1, mdl1, lbl1, ml1 = load_model("stage1_gatekeeper")
train_stage1_probs = predict_probs(
    train_df[TEXT_COL].fillna("").astype(str).tolist(), tok1, mdl1, ml1, BATCH_SIZE, desc="[train] stage1"
)
del mdl1, tok1
if DEVICE == "cuda":
    torch.cuda.empty_cache()

train_df["p_none"] = train_stage1_probs[:, 0]
train_df["p_labelled"] = train_stage1_probs[:, 1]

# Note: the OvR detectors are not needed on the full training set here — theta_none is
# tuned purely on Stage 1's own None-class separation, independent of the OvR stage.
print("Train rows scored:", len(train_df))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[train] stage1:   0%|          | 0/176 [00:00<?, ?it/s]

Train rows scored: 5608


In [48]:
def none_first_ovr_decision(p_none, ovr_pos_probs_df, ovr_thresholds, theta_none):
    cleared = pd.DataFrame({
        l: ovr_pos_probs_df[l] >= ovr_thresholds[l] for l in LEGAL_LABELS
    }, index=ovr_pos_probs_df.index)

    masked_probs = ovr_pos_probs_df.where(cleared, other=-1.0)
    any_cleared = cleared.any(axis=1)
    role_pred = masked_probs.idxmax(axis=1)
    role_pred = np.where(any_cleared, role_pred, "None")

    return pd.Series(np.where(p_none >= theta_none, "None", role_pred), index=p_none.index)


train_is_none = (train_df["label"] == "None").astype(int)

sweep_results = []
for theta in THETA_GRID:
    pred_is_none = (train_df["p_none"] >= theta).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        train_is_none, pred_is_none, labels=[0, 1], average="binary", pos_label=1, zero_division=0
    )
    sweep_results.append({
        "theta_none": theta,
        "none_precision": round(precision, 4),
        "none_recall": round(recall, 4),
        "none_f1": round(f1, 4)
    })

sweep_df = pd.DataFrame(sweep_results)
best_theta_f1 = sweep_df.loc[sweep_df["none_f1"].idxmax(), "theta_none"]
best_theta_precision = sweep_df.loc[sweep_df["none_precision"].idxmax(), "theta_none"]
best_theta_none = best_theta_f1

print(sweep_df.to_string(index=False))
print(f"\nBest theta_none maximizing None-class F1: {best_theta_f1}")
print(f"Best theta_none maximizing None-class precision: {best_theta_precision}")
print(f"Using F1-maximizing threshold going forward: {best_theta_none}")

 theta_none  none_precision  none_recall  none_f1
       0.05          0.5107       0.9715   0.6694
       0.10          0.6363       0.9349   0.7573
       0.15          0.7124       0.9029   0.7964
       0.20          0.7556       0.8778   0.8121
       0.25          0.8008       0.8518   0.8255
       0.30          0.8340       0.8252   0.8296
       0.35          0.8515       0.7982   0.8240
       0.40          0.8701       0.7717   0.8179
       0.45          0.8828       0.7471   0.8093
       0.50          0.8933       0.7211   0.7980
       0.55          0.9026       0.6915   0.7831
       0.60          0.9098       0.6620   0.7664
       0.65          0.9196       0.6355   0.7516
       0.70          0.9285       0.6044   0.7322
       0.75          0.9344       0.5709   0.7087
       0.80          0.9425       0.5418   0.6881
       0.85          0.9536       0.4832   0.6414
       0.90          0.9647       0.4106   0.5760
       0.95          0.9682       0.2899   0.4462


### 13.3 Apply to validation (transparency check) and test

In [ ]:
# Full validation set OvR probabilities (section 12 only scored the legal-relevant subset)
eval_ovr_probs_full = {}
for label in LEGAL_LABELS:
    clean = clean_col(label)
    model_key = f"ovr_{clean}"
    tokenizer, model, _, max_len = load_model(model_key)
    probs = predict_probs(
        eval_df[TEXT_COL].fillna("").astype(str).tolist(),
        tokenizer, model, max_len, BATCH_SIZE, desc=f"[val-full] {model_key}"
    )
    del model, tokenizer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    eval_ovr_probs_full[label] = pd.Series(probs[:, 1], index=eval_df.index)

eval_ovr_probs_df = pd.DataFrame(eval_ovr_probs_full)
eval_p_none_full = eval_stage1_text_probs[0]

val_pred_threshold = none_first_ovr_decision(eval_p_none_full, eval_ovr_probs_df, ovr_theta_class, best_theta_none)
_, _, val_f1, _ = precision_recall_fscore_support(
    eval_df["label"], val_pred_threshold, labels=ALL_LABELS, average="macro", zero_division=0
)
print(f"[Transparency check] Validation macro F1 at theta_none={best_theta_none}: {val_f1:.4f}")

test_ovr_probs_df = pd.DataFrame({l: test_df[f"{clean_col(l)}_p_pos"] for l in LEGAL_LABELS})
test_pred_threshold = none_first_ovr_decision(test_df["p_none"], test_ovr_probs_df, ovr_theta_class, best_theta_none)

acc = accuracy_score(test_df["label"], test_pred_threshold)
_, _, f1, _ = precision_recall_fscore_support(
    test_df["label"], test_pred_threshold, labels=ALL_LABELS, average="macro", zero_division=0
)

print(f"\nConfidence-thresholded Pipeline B (theta_none={best_theta_none}) on test set")
print(f"Accuracy: {acc:.4f}  Macro F1: {f1:.4f}")
print()
print(classification_report(test_df["label"], test_pred_threshold, labels=ALL_LABELS, digits=4, zero_division=0))


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[val-full] ovr_beoordeling:   0%|          | 0/41 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[val-full] ovr_beslissing:   0%|          | 0/41 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[val-full] ovr_materiele_feiten:   0%|          | 0/41 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[val-full] ovr_proceshandelingen:   0%|          | 0/41 [00:00<?, ?it/s]

[Transparency check] Validation macro F1 at theta_none=0.3: 0.6875

Confidence-thresholded Pipeline B (theta_none=0.3) on test set
Accuracy: 0.6418  Macro F1: 0.6534

                   precision    recall  f1-score   support

             None     0.6336    0.7449    0.6848       541
      beoordeling     0.6322    0.6761    0.6534       389
       beslissing     0.9000    0.7826    0.8372        69
 materiele feiten     0.6635    0.4714    0.5512       297
proceshandelingen     0.5810    0.5049    0.5403       206

         accuracy                         0.6418      1502
        macro avg     0.6821    0.6360    0.6534      1502
     weighted avg     0.6442    0.6418    0.6374      1502

                                  system  test_accuracy  test_macro_f1
0              Plain formula (no fusion)         0.6372         0.6604
1                  Header fusion (tuned)         0.6551         0.6810
2  Confidence threshold (theta_none=0.3)         0.6418         0.6534
